**Cell 1**

# NB13.2 — Vertex length test: does `max_new_tokens` explain the similar ArmoRM scores?

**Question.** In NB13.1, ArmoRM scores of all merge points are very similar. 79 % of the
answers end without a sentence end, i.e. they are cut at `max_new_tokens = 256`. This
notebook tests whether the similarity is caused by truncation or by the adapters.

**Design (diagnostic, exploratory, not a method comparison).**

* Same five NB11.1 mean-DPO adapters, same 80 NB13.1 prompts, same TinyLlama
  chat template, same greedy decoding, same float32 effective-delta merge, same ArmoRM
  (int8, golden sample checked).
* Points: the five vertices, the uniform point, and the base model θ₀ (no adapter).
* Conditions: `max_new_tokens = 256` and `512`.
  * 256 answers for vertices and uniform are **taken from the frozen NB13.1 answer cache**
    (hash-checked). A determinism spot check regenerates 8 of them first.
  * Only the 512 condition (7 points) and the base model at 256 are generated new.
* Both conditions are scored by ArmoRM **in the same session**. The rescored 256 answers are
  compared against the NB13.1 ArmoRM cache as a scorer check.
* Nothing in NB13.1 is changed. This notebook writes to its own `RUN_TAG`.

**Quantities.** Vertex matrix `M[adapter, head]`, number of adapters that lead on their own
head, column spread per head, diagonal dominance
`D = mean_k ( M[k,k] − mean_{j≠k} M[j,k] )`, own-head gain of each vertex against the uniform
point, and the between-point vs. within-prompt standard deviation.

**Reading rule, fixed before running.**

* **Length matters** if the paired bootstrap CI of `D(512) − D(256)` lies above 0 **and**
  the number of own-head leaders rises by at least 2.
* **Length helps, but not enough** if only the CI condition holds.
* **Similarity comes from the adapters** otherwise.

**Runtime (A100).** About 7 × 10–15 min generation at 512 tokens, about 5 min at 256, plus
about 15 min ArmoRM scoring. Everything is resumable from Drive.


**Cell 2**

Drive, repository and the NB11.1 mean-adapter bundle (identical to NB13.1 Cell 3).


In [ ]:
# Cell 3
# Drive is intentionally the first interactive action in "Run all".
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

%cd /content
import hashlib, json, os, shutil, stat, zipfile
from pathlib import Path

repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"
if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

ADAPTER_AXES = ("helpfulness", "correctness", "coherence", "complexity", "verbosity")
NB11_1_RUN_TAG = "nb11_1_multiseed_pair137_train137_141"
BUNDLE_NAME = "nb11_1_tinyllama_helpsteer2_dpo_multiseed_mean_adapters.zip"
drive_bundle = (
    Path("/content/drive/MyDrive/master-thesis-nb11-1")
    / NB11_1_RUN_TAG / BUNDLE_NAME
)
candidates = [path for path in (
    drive_bundle,
    Path("/content") / BUNDLE_NAME,
    Path.cwd() / BUNDLE_NAME,
) if path.is_file()]
if not candidates:
    from google.colab import files
    print(f"Bitte jetzt {BUNDLE_NAME} auswählen.")
    uploaded = files.upload()
    candidates = [Path.cwd() / name for name in uploaded if name == BUNDLE_NAME]
if not candidates:
    raise FileNotFoundError(f"Required NB11.1 bundle not found: {BUNDLE_NAME}")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

# Duplicate copies are harmless only when their bytes agree.
candidate_hashes = {str(path.resolve()): sha256_file(path) for path in candidates}
if len(set(candidate_hashes.values())) != 1:
    raise RuntimeError(f"Conflicting NB11.1 bundles found: {candidate_hashes}")
ADAPTER_BUNDLE_PATH = candidates[0].resolve()
ADAPTER_BUNDLE_SHA256 = next(iter(candidate_hashes.values()))
ADAPTER_INPUT_ROOT = Path("/content/nb13_1_inputs") / ADAPTER_BUNDLE_SHA256[:16] / "mean_adapters"
RUN_MANIFEST_PATH = ADAPTER_INPUT_ROOT.parent / "geometry/nb11_1_run_manifest.json"

required_members = {
    "geometry/nb11_1_run_manifest.json",
    *{
        f"mean_adapters/dpo_{axis}/adapter/{filename}"
        for axis in ADAPTER_AXES
        for filename in (
            "adapter_config.json", "adapter_model.safetensors", "averaging_manifest.json"
        )
    },
}
required_on_disk = [ADAPTER_INPUT_ROOT.parent / member for member in sorted(required_members)]
if not all(path.is_file() for path in required_on_disk):
    with zipfile.ZipFile(ADAPTER_BUNDLE_PATH) as bundle:
        entries = {entry.filename: entry for entry in bundle.infolist()}
        missing = sorted(required_members - set(entries))
        if missing:
            raise FileNotFoundError(f"NB11.1 bundle is incomplete; missing={missing}")
        payload_size = sum(entries[name].file_size for name in required_members)
        if payload_size > 1_500_000_000:
            raise RuntimeError(f"Unexpectedly large adapter payload: {payload_size} bytes")
        target = ADAPTER_INPUT_ROOT.parent.resolve()
        target.mkdir(parents=True, exist_ok=True)
        for name in sorted(required_members):
            entry = entries[name]
            unix_mode = (entry.external_attr >> 16) & 0o170000
            if unix_mode == stat.S_IFLNK:
                raise RuntimeError(f"Symlink forbidden in adapter ZIP: {name}")
            destination = (target / name).resolve()
            if target != destination and target not in destination.parents:
                raise RuntimeError(f"Unsafe path in adapter ZIP: {name}")
            destination.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(entry) as source, destination.open("wb") as output:
                shutil.copyfileobj(source, output)

missing = [str(path) for path in required_on_disk if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Extraction incomplete: {missing}")

run_manifest = json.loads(RUN_MANIFEST_PATH.read_text(encoding="utf-8"))
assert run_manifest["run_tag"] == NB11_1_RUN_TAG
assert run_manifest["training_seeds"] == [137, 138, 139, 140, 141]
assert run_manifest["pair_seed"] == 137
assert run_manifest["attribute_order"] == list(ADAPTER_AXES)
assert run_manifest["mean_operation"] == "exact arithmetic mean of effective LoRA updates"
assert run_manifest["mean_output_rank"] == 40

MEAN_MANIFESTS = {}
ADAPTER_PATHS = {}
for axis in ADAPTER_AXES:
    adapter = ADAPTER_INPUT_ROOT / f"dpo_{axis}/adapter"
    manifest_path = adapter / "averaging_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    config = json.loads((adapter / "adapter_config.json").read_text(encoding="utf-8"))
    assert manifest["operation"] == "exact_arithmetic_mean_of_effective_lora_updates"
    assert manifest["label"] == axis
    assert manifest["source_count"] == 5
    assert [entry["seed"] for entry in manifest["source_adapters"]] == [137, 138, 139, 140, 141]
    assert manifest["source_rank"] == 8 and manifest["output_rank"] == 40
    assert manifest["output_lora_alpha"] == 16
    assert manifest["verification_relative_frobenius_error"] <= 1e-6
    assert config["r"] == 40 and config["lora_alpha"] == 16
    assert manifest["adapter_model_sha256"] == sha256_file(adapter / "adapter_model.safetensors")
    assert manifest["adapter_config_sha256"] == sha256_file(adapter / "adapter_config.json")
    assert run_manifest["mean_adapter_hashes"][axis] == manifest["adapter_model_sha256"]
    MEAN_MANIFESTS[axis] = manifest
    ADAPTER_PATHS[axis] = adapter.resolve()
    print(f"[OK] mean {axis:12s} {manifest['adapter_model_sha256'][:16]}...")
print(f"[OK] NB11.1 bundle SHA256 = {ADAPTER_BUNDLE_SHA256}")


**Cell 4**

GPU check.


In [ ]:
# Cell 5
!nvidia-smi || echo "No local GPU. Generation and ArmoRM require one."


**Cell 6**

Pinned runtime (identical to NB13.1 Cell 6).


In [ ]:
# Cell 7
import importlib.metadata, subprocess, sys

print(f"Python {sys.version.split()[0]} ({sys.executable})")
packages = [
    "pandas==2.2.3", "numpy==2.1.3", "scipy==1.14.1",
    "protobuf==5.29.5", "transformers==4.45.2", "tokenizers==0.20.3",
    "peft==0.13.2", "accelerate==1.1.1", "datasets==3.1.0",
    "huggingface_hub==0.36.0", "bitsandbytes==0.50.2", "pyyaml", "safetensors", "psutil",
    "tritonclient[http]==2.60.0",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "--only-binary=:all:", *packages],
    check=True,
)
expected = {"transformers": "4.45.2", "tokenizers": "0.20.3", "peft": "0.13.2",
            "tritonclient": "2.60.0"}
installed = {name: importlib.metadata.version(name) for name in expected}
assert installed == expected, f"Unexpected inference stack: {installed}"
print("Runtime:", {name: importlib.metadata.version(name) for name in
       ("transformers", "tokenizers", "peft", "accelerate", "datasets", "tritonclient")})


**Cell 8**

Configuration. All decoding settings except `max_new_tokens` are copied from NB13.1 and asserted against its preregistration.


In [ ]:
# Cell 9
from __future__ import annotations

# Colab ships a TensorFlow built against protobuf 6, while the pinned stack uses protobuf 5.29.5.
# transformers would import TensorFlow inside every ArmoRM forward pass (is_tensor check) and crash.
# This must be set before transformers is imported for the first time.
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

import gc, time
from contextlib import contextmanager
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path("/content/master-thesis").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RUN_TAG = "nb13_2_vertex_length_test_run1"
CLAIM_STATUS = "exploratory_diagnostic"
RESULTS_DIR = PROJECT_ROOT / "results/nb13_2_vertex_length_test" / RUN_TAG
PERSISTENT_DIR = Path("/content/drive/MyDrive/master-thesis-nb13-2") / RUN_TAG
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PERSISTENT_DIR.mkdir(parents=True, exist_ok=True)
ANSWER_CACHE = PERSISTENT_DIR / "answer_cache.jsonl"
SCORE_CACHE = PERSISTENT_DIR / "armorm_scores.jsonl"
REPORT_JSON = RESULTS_DIR / "nb13_2_report.json"

# Frozen NB13.1 reference (committed in the repository).
NB13_1_DIR = PROJECT_ROOT / "results/nb13_1_nb11_1_mean_dpo_three_rms_run1_two_rms_provisional_outputs"
NB13_1_FILES = {
    "nb13_1_reward_prompts.jsonl": "47eada70d0430a41a4506f43cc26950f14657d78d3d351a0d951f8af43161055",
    "answer_cache.jsonl": "bb8f260b2e2c635cb7d15369d065ba11ff5c2245d95efec46ee3380d25ffdf2b",
    "reward_cache_armorm.jsonl": "7599805d8711c0febe9c2993e8d012e69438fd9d140f6942f57186ced9c09a54",
}
NB13_1_ADAPTER_BUNDLE_SHA256 = "5b75c3d98ec94e05a8dab58fbec99d9da8c08ecd09ba35f2f14f6838ee6818fc"

TOKEN_CONDITIONS = (256, 512)
REUSED_CONDITION = 256           # vertices + uniform come from NB13.1 at this length
REPETITION_PENALTY = 1.15
NO_REPEAT_NGRAM_SIZE = 5
PROMPT_MAX_LENGTH = 512
DETERMINISM_CHECK_N = 8
N_BOOT = 10_000
BOOT_SEED = 1313

ARMORM_MODEL = "RLHFlow/ArmoRM-Llama3-8B-v0.1"
ARMORM_REVISION = "eb2676d20da2f2d41082289d23c59b9f7427f955"

ATTRIBUTES = list(ADAPTER_AXES)
m = len(ATTRIBUTES)
BASE_MODEL_NAME = run_manifest["base_model"]
BASE_REVISION = run_manifest["base_revision"]
assert BASE_MODEL_NAME == "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
assert BASE_REVISION == "fe8a4ea1ffedaf415f4da2f062534de366a451e6"
assert ADAPTER_BUNDLE_SHA256 == NB13_1_ADAPTER_BUNDLE_SHA256, "Not the NB13.1 adapter bundle."

for name, expected in NB13_1_FILES.items():
    observed = sha256_file(NB13_1_DIR / name)
    assert observed == expected, f"{name}: {observed} != frozen NB13.1 hash"

nb13_1_prereg = json.loads((NB13_1_DIR / "nb13_1_preregistration.json").read_text(encoding="utf-8"))
gen = nb13_1_prereg["binding"]["generation"]
assert gen["max_new_tokens"] == REUSED_CONDITION
assert gen["repetition_penalty"] == REPETITION_PENALTY
assert gen["no_repeat_ngram_size"] == NO_REPEAT_NGRAM_SIZE
assert gen["merge_dtype"] == "float32"
assert nb13_1_prereg["binding"]["reward_models"]["armorm"]["revision"] == ARMORM_REVISION
print(f"run tag    = {RUN_TAG}\nresults    = {RESULTS_DIR}\npersistent = {PERSISTENT_DIR}")


**Cell 10**

Prompts, frozen NB13.1 answers, and the seven evaluation points.


In [ ]:
# Cell 11
from src.proxy_validation import coefficient_key, write_json

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

reward_prompts = read_jsonl(NB13_1_DIR / "nb13_1_reward_prompts.jsonl")
PROMPTS = [row["prompt"] for row in reward_prompts]
N_PROMPTS = len(PROMPTS)
assert N_PROMPTS == 80

nb13_1_answers = {row["key"]: row for row in read_jsonl(NB13_1_DIR / "answer_cache.jsonl")[1:]}
nb13_1_armorm = {row["key"]: np.asarray(row["scores"], float)
                 for row in read_jsonl(NB13_1_DIR / "reward_cache_armorm.jsonl")[1:]}

POINTS = {"base": np.zeros(m), "uniform": np.full(m, 1.0 / m)}
for k, axis in enumerate(ATTRIBUTES):
    POINTS[f"vertex_{axis}"] = np.eye(m)[k]
POINT_ORDER = list(POINTS)
VERTICES = [f"vertex_{axis}" for axis in ATTRIBUTES]
ADAPTER_POINTS = ["uniform", *VERTICES]

REUSED = {}
for name in ADAPTER_POINTS:
    record = nb13_1_answers[coefficient_key(POINTS[name])]
    assert [row["prompt_id"] for row in reward_prompts] == record["prompt_ids"]
    assert len(record["answers"]) == N_PROMPTS
    REUSED[name] = record["answers"]
print(f"Reused NB13.1 answers at {REUSED_CONDITION} tokens for: {list(REUSED)}")

BINDING = {
    "run_tag": RUN_TAG, "claim_status": CLAIM_STATUS,
    "adapter_bundle_sha256": ADAPTER_BUNDLE_SHA256,
    "nb13_1_files": NB13_1_FILES,
    "points": {name: vec.tolist() for name, vec in POINTS.items()},
    "token_conditions": list(TOKEN_CONDITIONS),
    "reused_condition": REUSED_CONDITION,
    "generation": {"repetition_penalty": REPETITION_PENALTY, "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
                   "decoding": "greedy; do_sample=False; num_beams=1", "prompt_max_length": PROMPT_MAX_LENGTH,
                   "prompt_format": "TinyLlama tokenizer chat template", "merge_dtype": "float32"},
    "armorm": {"name": ARMORM_MODEL, "revision": ARMORM_REVISION, "precision": "int8"},
    "source_sha256": {name: sha256_file(PROJECT_ROOT / "src" / name)
                      for name in ("merge.py", "armorm_scorer.py", "armorm_objectives.py", "effective_lora_geometry.py")},
}
BINDING_SHA256 = hashlib.sha256(json.dumps(BINDING, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
print(f"binding SHA256 = {BINDING_SHA256}")


**Cell 12**

TinyLlama generation model with the exact float32 effective-delta merge (same code path as NB13.1 Cell 23). `generate_answer` additionally returns the number of new tokens and whether EOS was reached.


In [ ]:
# Cell 13
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.merge import combine_effective_deltas, effective_deltas, resolve_base_module

generation_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, revision=BASE_REVISION, use_fast=True)
generation_tokenizer.pad_token = generation_tokenizer.eos_token
generation_tokenizer.padding_side = "left"
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, revision=BASE_REVISION, torch_dtype=torch.float32, device_map="auto"
).eval()
DELTAS = effective_deltas(dict(ADAPTER_PATHS))
assert list(DELTAS) == ATTRIBUTES

@contextmanager
def merged_model(model, deltas_by_adapter, lam):
    if np.allclose(lam, 0.0):          # base model theta_0: nothing to add
        yield model
        return
    merged = combine_effective_deltas(lam, deltas_by_adapter)
    originals = {}
    try:
        for module_name, delta_cpu in merged.items():
            module = resolve_base_module(model, module_name)
            originals[module_name] = module.weight.detach().clone()
            update = module.weight.detach().float() + delta_cpu.to(module.weight.device, torch.float32)
            with torch.no_grad():
                module.weight.copy_(update.to(dtype=module.weight.dtype))
        yield model
    finally:
        for module_name, original in originals.items():
            with torch.no_grad():
                resolve_base_module(model, module_name).weight.copy_(original)
        del originals, merged
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def generate_answer(prompt, max_new_tokens):
    device = next(base_model.parameters()).device
    rendered = generation_tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    encoded = generation_tokenizer(rendered, return_tensors="pt", add_special_tokens=False,
                                   truncation=True, max_length=PROMPT_MAX_LENGTH)
    input_ids = encoded["input_ids"].to(device)
    with torch.inference_mode():
        generated = base_model.generate(
            input_ids=input_ids, attention_mask=encoded["attention_mask"].to(device),
            max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
            repetition_penalty=REPETITION_PENALTY, no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
            pad_token_id=generation_tokenizer.pad_token_id,
            eos_token_id=generation_tokenizer.eos_token_id,
        )
    new_tokens = generated[0, input_ids.shape[1]:]
    reached_eos = bool((new_tokens == generation_tokenizer.eos_token_id).any())
    text = generation_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return text, int(new_tokens.shape[0]), reached_eos
print("Generation model and exact mean-adapter deltas are ready.")


**Cell 14**

Determinism spot check: regenerate the first 8 prompts of the helpfulness vertex at 256 tokens and compare with the frozen NB13.1 answers. A mismatch does not stop the run, but it is reported, because then the reused 256 answers and the new 512 answers come from slightly different numerics.


In [ ]:
# Cell 15
with merged_model(base_model, DELTAS, POINTS["vertex_helpfulness"]):
    regenerated = [generate_answer(p, REUSED_CONDITION)[0] for p in PROMPTS[:DETERMINISM_CHECK_N]]
frozen = REUSED["vertex_helpfulness"][:DETERMINISM_CHECK_N]
DETERMINISM = {"n_checked": DETERMINISM_CHECK_N,
               "n_identical": int(sum(a == b for a, b in zip(regenerated, frozen)))}
print(DETERMINISM)
if DETERMINISM["n_identical"] < DETERMINISM_CHECK_N:
    print("WARNING: regeneration differs from NB13.1 (GPU type or library numerics). "
          "Interpret 256 vs 512 differences with this in mind.")


**Cell 16**

Generation. New: all seven points at 512 tokens, and the base model at 256 tokens. Every finished point is written to Drive immediately, so the cell can be re-run after a disconnect.


In [ ]:
# Cell 17
def load_cache(path, header):
    if not path.is_file() or path.stat().st_size == 0:
        path.write_text(json.dumps(header) + "\n", encoding="utf-8")
        return {}
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert rows[0] == header, f"{path.name} belongs to a different binding. Use a new RUN_TAG."
    return {row["key"]: row for row in rows[1:]}

answer_header = {"kind": "nb13_2_answer_cache", "binding_sha256": BINDING_SHA256}
answer_records = load_cache(ANSWER_CACHE, answer_header)
JOBS = [(512, name) for name in POINT_ORDER] + [(256, "base")]

started = time.perf_counter()
with ANSWER_CACHE.open("a", encoding="utf-8") as handle:
    for done, (max_new, name) in enumerate(JOBS, start=1):
        key = f"{max_new}|{name}"
        if key in answer_records:
            continue
        t0 = time.perf_counter()
        with merged_model(base_model, DELTAS, POINTS[name]):
            outputs = [generate_answer(prompt, max_new) for prompt in PROMPTS]
        record = {"key": key, "max_new_tokens": max_new, "point": name, "lambda": POINTS[name].tolist(),
                  "answers": [o[0] for o in outputs], "n_new_tokens": [o[1] for o in outputs],
                  "reached_eos": [o[2] for o in outputs]}
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush(); os.fsync(handle.fileno())
        answer_records[key] = record
        print(f"[generate] {done}/{len(JOBS)} {key:28s} {time.perf_counter()-t0:6.1f}s "
              f"eos={np.mean(record['reached_eos']):.0%}")

ANSWERS = {}
for max_new, name in [(c, n) for c in TOKEN_CONDITIONS for n in POINT_ORDER]:
    key = f"{max_new}|{name}"
    if key in answer_records:
        ANSWERS[(max_new, name)] = answer_records[key]["answers"]
    else:
        assert max_new == REUSED_CONDITION and name in REUSED
        ANSWERS[(max_new, name)] = REUSED[name]
assert len(ANSWERS) == len(TOKEN_CONDITIONS) * len(POINT_ORDER)
ANSWER_CACHE_SHA256 = sha256_file(ANSWER_CACHE)
print(f"[OK] answer sets: {len(ANSWERS)}; elapsed {timedelta(seconds=int(time.perf_counter()-started))}")

del base_model, DELTAS
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Generation model released before ArmoRM is loaded.")


**Cell 18**

ArmoRM scoring of all 14 answer sets in one session. The six reused 256 sets are scored again and compared with the NB13.1 ArmoRM cache.


In [ ]:
# Cell 19
import transformers.utils.generic as _hf_generic
from transformers.utils import import_utils as _hf_import_utils
# Second safeguard, also effective if transformers was imported before USE_TF was set:
_hf_generic.is_tf_available = lambda: False
_hf_import_utils.is_tf_available = lambda: False

import logging

class _OnceFilter(logging.Filter):
    """Let each distinct message through once; bitsandbytes repeats one cast notice per forward pass."""
    def __init__(self):
        super().__init__()
        self._seen = set()
    def filter(self, record):
        message = record.getMessage()
        if message in self._seen:
            return False
        self._seen.add(message)
        return True

_bnb_logger = logging.getLogger("bitsandbytes.autograd._functions")
if not any(type(f).__name__ == "_OnceFilter" for f in _bnb_logger.filters):
    _bnb_logger.addFilter(_OnceFilter())

from src.armorm_scorer import make_score_prompt_answer

armorm_score, armorm = make_score_prompt_answer(
    ARMORM_MODEL, revision=ARMORM_REVISION, dtype="bfloat16", load_in_8bit=True
)
armorm.assert_golden_sample()

score_header = {"kind": "nb13_2_armorm_scores", "binding_sha256": BINDING_SHA256,
                "answer_cache_sha256": ANSWER_CACHE_SHA256}
score_records = load_cache(SCORE_CACHE, score_header)
with SCORE_CACHE.open("a", encoding="utf-8") as handle:
    for (max_new, name), answers in ANSWERS.items():
        key = f"{max_new}|{name}"
        if key in score_records:
            continue
        t0 = time.perf_counter()
        scores = np.asarray([armorm_score(p, a, ATTRIBUTES) for p, a in zip(PROMPTS, answers)], float)
        assert scores.shape == (N_PROMPTS, m) and np.all(np.isfinite(scores))
        record = {"key": key, "scores": scores.tolist()}
        handle.write(json.dumps(record) + "\n")
        handle.flush(); os.fsync(handle.fileno())
        score_records[key] = record
        print(f"[armorm] {key:28s} mean={np.round(scores.mean(0), 4)} {time.perf_counter()-t0:5.1f}s")

S = {c: np.stack([np.asarray(score_records[f"{c}|{n}"]["scores"], float) for n in POINT_ORDER])
     for c in TOKEN_CONDITIONS}                                   # S[c]: (points, prompts, heads)

SCORER_CHECK = {}
for name in ADAPTER_POINTS:
    frozen_scores = nb13_1_armorm[coefficient_key(POINTS[name])]
    SCORER_CHECK[name] = float(np.abs(S[REUSED_CONDITION][POINT_ORDER.index(name)] - frozen_scores).max())
print("max |rescored 256 - NB13.1 ArmoRM| per point:", {k: round(v, 5) for k, v in SCORER_CHECK.items()})

armorm.unload() if hasattr(armorm, "unload") else None
del armorm_score, armorm
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


**Cell 20**

Answer length and truncation. Token counts are recomputed with the generation tokenizer for every answer, so both conditions are measured the same way (reused NB13.1 answers carry no stored token counts).


In [ ]:
# Cell 21
SENTENCE_END = (".", "!", "?", '"', ")", "`", "|", "*", ":")

def n_tokens(text):
    return len(generation_tokenizer(text, add_special_tokens=False)["input_ids"])

length_rows = []
for (max_new, name), answers in ANSWERS.items():
    counts = np.asarray([n_tokens(a) for a in answers])
    length_rows.append({
        "max_new_tokens": max_new, "point": name,
        "mean_tokens": counts.mean(), "median_tokens": np.median(counts),
        "share_at_limit": float(np.mean(counts >= max_new - 3)),
        "share_no_sentence_end": float(np.mean([not a.rstrip().endswith(SENTENCE_END) for a in answers])),
        "mean_words": float(np.mean([len(a.split()) for a in answers])),
    })
LENGTH_DF = pd.DataFrame(length_rows)
LENGTH_DF.to_csv(RESULTS_DIR / "length_stats.csv", index=False)
display(LENGTH_DF.pivot(index="point", columns="max_new_tokens",
                        values=["mean_tokens", "share_at_limit", "share_no_sentence_end"]).round(3))


**Cell 22**

Vertex matrix, own-head leaders, column spread, diagonal dominance, and the variance decomposition, per condition.


In [ ]:
# Cell 23
IDX = {name: i for i, name in enumerate(POINT_ORDER)}
V_IDX = [IDX[v] for v in VERTICES]

def vertex_matrix(tensor, prompts=slice(None)):
    return tensor[V_IDX][:, prompts, :].mean(axis=1)            # (adapter, head)

def diag_dominance(M):
    off = (M.sum(axis=0) - np.diag(M)) / (m - 1)
    return float(np.mean(np.diag(M) - off))

def describe_condition(c):
    M = vertex_matrix(S[c])
    head_means = S[c].mean(axis=2)                              # (points, prompts)
    adapters = [IDX[n] for n in ADAPTER_POINTS]
    return M, {
        "own_head_leaders": int(sum(M[:, k].argmax() == k for k in range(m))),
        "leader_per_head": {ATTRIBUTES[k]: ATTRIBUTES[int(M[:, k].argmax())] for k in range(m)},
        "column_spread_per_head": dict(zip(ATTRIBUTES, (M.max(0) - M.min(0)).round(5).tolist())),
        "column_spread_mean": float((M.max(0) - M.min(0)).mean()),
        "diag_dominance": diag_dominance(M),
        "std_within_prompts": float(head_means[adapters].std(axis=1).mean()),
        "std_between_points": float(head_means[adapters].mean(axis=1).std()),
    }

SUMMARY = {}
for c in TOKEN_CONDITIONS:
    M, SUMMARY[c] = describe_condition(c)
    pd.DataFrame(M, index=ATTRIBUTES, columns=ATTRIBUTES).to_csv(RESULTS_DIR / f"vertex_matrix_{c}.csv")
    print(f"\n=== max_new_tokens = {c}: vertex matrix (row = adapter, column = ArmoRM head) ===")
    display(pd.DataFrame(M, index=ATTRIBUTES, columns=ATTRIBUTES).round(4))
display(pd.DataFrame(SUMMARY).drop(["leader_per_head", "column_spread_per_head"]).round(5))


**Cell 24**

Paired bootstrap over the 80 prompts: own-head gain of each vertex against the uniform point, and the change of diagonal dominance and column spread from 256 to 512 tokens.


In [ ]:
# Cell 25
rng = np.random.default_rng(BOOT_SEED)
boot_idx = rng.integers(0, N_PROMPTS, size=(N_BOOT, N_PROMPTS))

def ci(values):
    lo, hi = np.percentile(values, [2.5, 97.5])
    return float(lo), float(hi)

own_rows = []
for c in TOKEN_CONDITIONS:
    for k, axis in enumerate(ATTRIBUTES):
        d = S[c][IDX[f"vertex_{axis}"], :, k] - S[c][IDX["uniform"], :, k]
        boots = d[boot_idx].mean(axis=1)
        lo, hi = ci(boots)
        own_rows.append({"max_new_tokens": c, "adapter": axis, "own_head_gain_vs_uniform": d.mean(),
                         "ci_low": lo, "ci_high": hi, "excludes_zero": bool(lo > 0 or hi < 0)})
OWN_DF = pd.DataFrame(own_rows)
OWN_DF.to_csv(RESULTS_DIR / "own_head_gain_vs_uniform.csv", index=False)
display(OWN_DF.round(4))

def stat_diff(fn, prompts):
    return fn(vertex_matrix(S[512], prompts)) - fn(vertex_matrix(S[256], prompts))

spread = lambda M: float((M.max(0) - M.min(0)).mean())
dd_boot = np.array([stat_diff(diag_dominance, b) for b in boot_idx])
sp_boot = np.array([stat_diff(spread, b) for b in boot_idx])
DELTAS_512_MINUS_256 = {
    "diag_dominance": {"estimate": stat_diff(diag_dominance, slice(None)), "ci": ci(dd_boot)},
    "column_spread_mean": {"estimate": stat_diff(spread, slice(None)), "ci": ci(sp_boot)},
    "own_head_leaders": SUMMARY[512]["own_head_leaders"] - SUMMARY[256]["own_head_leaders"],
}
print(json.dumps(DELTAS_512_MINUS_256, indent=2))


**Cell 26**

Base model θ₀ against the adapters: how far do the adapters move the model at all?


In [ ]:
# Cell 27
base_rows = []
for c in TOKEN_CONDITIONS:
    base = S[c][IDX["base"]]
    for name in ADAPTER_POINTS:
        d = S[c][IDX[name]].mean(axis=1) - base.mean(axis=1)
        lo, hi = ci(d[boot_idx].mean(axis=1))
        base_rows.append({
            "max_new_tokens": c, "point": name,
            "identical_answers_to_base": int(sum(a == b for a, b in zip(ANSWERS[(c, name)], ANSWERS[(c, "base")]))),
            "head_mean_minus_base": float(d.mean()), "ci_low": lo, "ci_high": hi,
        })
BASE_DF = pd.DataFrame(base_rows)
BASE_DF.to_csv(RESULTS_DIR / "adapters_vs_base.csv", index=False)
display(BASE_DF.round(4))


**Cell 28**

Verdict according to the reading rule fixed in Cell 1.


In [ ]:
# Cell 29
ci_low, ci_high = DELTAS_512_MINUS_256["diag_dominance"]["ci"]
ci_positive = ci_low > 0
leaders_up = DELTAS_512_MINUS_256["own_head_leaders"] >= 2
if ci_positive and leaders_up:
    VERDICT = "length_matters"
elif ci_positive:
    VERDICT = "length_helps_but_not_enough"
else:
    VERDICT = "similarity_comes_from_adapters"
print(f"Diagonal dominance 256 -> 512: {SUMMARY[256]['diag_dominance']:+.4f} -> {SUMMARY[512]['diag_dominance']:+.4f} "
      f"(diff CI [{ci_low:+.4f}, {ci_high:+.4f}])")
print(f"Own-head leaders 256 -> 512:   {SUMMARY[256]['own_head_leaders']} -> {SUMMARY[512]['own_head_leaders']}")
print(f"VERDICT: {VERDICT}")


**Cell 30**

Report and export. Results go to the repository results folder, a copy of the ZIP to Drive.


In [ ]:
# Cell 31
report = {
    "schema_version": 1, "notebook": "NB13.2 vertex length test", "run_tag": RUN_TAG,
    "created_utc": datetime.now(timezone.utc).isoformat(), "claim_status": CLAIM_STATUS,
    "binding": BINDING, "binding_sha256": BINDING_SHA256,
    "answer_cache_sha256": ANSWER_CACHE_SHA256,
    "determinism_check": DETERMINISM, "scorer_check_max_abs_diff": SCORER_CHECK,
    "length": LENGTH_DF.to_dict(orient="records"),
    "summary_by_condition": {str(c): SUMMARY[c] for c in TOKEN_CONDITIONS},
    "delta_512_minus_256": DELTAS_512_MINUS_256,
    "own_head_gain_vs_uniform": OWN_DF.to_dict(orient="records"),
    "adapters_vs_base": BASE_DF.to_dict(orient="records"),
    "reading_rule": "length_matters iff CI(D512-D256) > 0 and own-head leaders +2; "
                    "length_helps_but_not_enough iff only the CI holds; else similarity_comes_from_adapters",
    "verdict": VERDICT,
}
write_json(REPORT_JSON, report)

for path in (ANSWER_CACHE, SCORE_CACHE):
    shutil.copy2(path, RESULTS_DIR / path.name)
zip_path = RESULTS_DIR / f"{RUN_TAG}_outputs.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS_DIR.iterdir()):
        if path.is_file() and path != zip_path:
            archive.write(path, path.name)
shutil.copy2(zip_path, PERSISTENT_DIR / zip_path.name)
print(f"ZIP: {zip_path}\nSHA256: {sha256_file(zip_path)}\nVERDICT: {VERDICT}")

from google.colab import files
files.download(str(zip_path))


**Cell 32**

Repository status.


In [ ]:
# Cell 33
!git status --short
